# Exp 29 - Abstention and escalation behaviour (development, $0, evaluator-side)
**Question, distinct from Exp 28's security audit:** does the system know when it lacks enough evidence, under ordinary ambiguity? Reuses saved predictions and the run log; no new LLM spend.

**Risk-coverage table, per system:** auto-resolution rate, accuracy on auto-resolved cases, escalation rate (= Human Review Rate), missed-escalation rate, over-escalation rate, false approvals among auto-resolved cases.

**R0 vs R1 vs R2 (Exp 12B):** errors captured by escalation vs. correct decisions unnecessarily escalated -- the number that decides whether abstention is worth its human-review cost.

In [1]:
import json, sys
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parents[1]; sys.path.insert(0, str(ROOT))
from src import config as C, evaluate
pd.set_option('display.width', 260, 'display.max_columns', 40, 'display.max_colwidth', 100, 'display.max_rows', 200)
gt = evaluate.load_gt()
def load(p): return [json.loads(l) for l in Path(p).read_text().splitlines()]
log = [json.loads(l) for l in (C.SHARED/'run_log.jsonl').read_text().splitlines()]
by_run = {}
for r in log:
    if r.get('type') == 'evaluation': by_run.setdefault(r['run_id'], []).append(r)
SYS = {
    '2c rules (Exp 2)': load(C.RESULTS/'development'/'exp02_rules'/'predictions_2c_regex_development.jsonl'),
    'H0 RAG+M4 (Exp 9)': load(C.RESULTS/'development'/'exp09_metadata'/'predictions_openai_gpt-4o-mini.jsonl'),
    'H1 arithmetic': [{'case_id': r['case_id'], 'predicted_decision': r['predicted_decision']} for r in by_run['EXP12_HYBRID-DEVELOPMENT-gpt-4o-mini-0b530b']],
    'H3 +evidence validation': [{'case_id': r['case_id'], 'predicted_decision': r['predicted_decision']} for r in by_run['EXP12_HYBRID-DEVELOPMENT-gpt-4o-mini-cdd848']],
    'H5 full hybrid (Exp 12)': load(C.RESULTS/'development'/'exp12_hybrid'/'predictions_openai_gpt-4o-mini.jsonl'),
    'workflow_v2 (Exp 18)': load(C.RESULTS/'development'/'exp18_fixed_workflow'/'predictions.jsonl'),
}
print(len(SYS), 'systems loaded from saved predictions and the run log (H1/H3 recovered by run_id, since only the last H-level of each notebook run was saved to disk)')

6 systems loaded from saved predictions and the run log (H1/H3 recovered by run_id, since only the last H-level of each notebook run was saved to disk)


## R0 / R1 / R2 (Exp 12B): reconstructed exactly as in Exp 13/16/18/19 (deterministic path + LLM residual)

In [2]:
from src import llm_exp, rules_text as RT, rules_v2 as RV2
cases = llm_exp.cases_for('DEVELOPMENT')
NEEDED = {'MEAL_CLIENT': ['attendees_total','external_names'], 'MEAL_EMPLOYEE': ['attendees_total'], 'HOTEL': ['nights'], 'AIRFARE': ['cabin','flight_hours'], 'MILEAGE': ['distance_km'],
          'GIFT': ['recipient_name','recipient_org'], 'SOFTWARE': ['business_owner'], 'TRAINING': ['learning_plan_id'], 'GROUND_TRANSPORT': ['origin','destination'], 'TELECOM': [], 'EQUIPMENT': [], 'CONFERENCE_FEE': [], 'CAR_RENTAL': [], 'OTHER': []}
def det_predict(cc):
    f = RT.parse(cc); c2 = dict(cc, form=f); d = RV2.decide(c2)
    unresolved = any(f.get(k) is None for k in NEEDED.get(f.get('expense_type'), [])); fallback = d['reason'] == 'No rule triggered.'
    return d, (not fallback and not unresolved)
DET = {cc['case_id']: det_predict(cc) for cc in cases}
llm_residual = {r['case_id']: r for r in load(C.RESULTS/'development'/'exp12b_resolver'/'predictions_openai_gpt-4o-mini.jsonl')}
R0 = [{'case_id': cc['case_id'], 'predicted_decision': DET[cc['case_id']][0]['decision']} for cc in cases]                                             # forced answer, always
R1 = [{'case_id': cc['case_id'], 'predicted_decision': (DET[cc['case_id']][0]['decision'] if DET[cc['case_id']][1] else 'ESCALATE')} for cc in cases]  # escalate all unresolved
R2 = [{'case_id': cc['case_id'], 'predicted_decision': (DET[cc['case_id']][0]['decision'] if DET[cc['case_id']][1] else llm_residual[cc['case_id']]['predicted_decision'])} for cc in cases]
SYS['R0 forced answer (Exp 12B)'] = R0; SYS['R1 escalate-unresolved (Exp 12B)'] = R1; SYS['R2 LLM residual (Exp 12B)'] = R2
residual_ids = [cid for cid in DET if not DET[cid][1]]; print(f'{len(residual_ids)} of 70 claims are the internally-unresolved subset shared by R0/R1/R2')

36 of 70 claims are the internally-unresolved subset shared by R0/R1/R2


## Risk-coverage table: every system

In [3]:
def risk_coverage(recs):
    m = {r['case_id']: r for r in recs}; n = len(cases)
    auto = [cid for cid in m if m[cid]['predicted_decision'] != 'ESCALATE']; esc = [cid for cid in m if m[cid]['predicted_decision'] == 'ESCALATE']
    auto_correct = sum(m[cid]['predicted_decision'] == gt[cid]['expected_decision'] for cid in auto)
    missed_esc = [cid for cid in m if gt[cid]['expected_decision'] == 'ESCALATE' and m[cid]['predicted_decision'] != 'ESCALATE']
    over_esc = [cid for cid in esc if gt[cid]['expected_decision'] != 'ESCALATE']
    gold_esc = [cid for cid in m if gt[cid]['expected_decision'] == 'ESCALATE']
    fa_auto = sum(m[cid]['predicted_decision'] == 'APPROVE' and gt[cid]['expected_decision'] != 'APPROVE' for cid in auto)
    non_approvable_auto = sum(gt[cid]['expected_decision'] != 'APPROVE' for cid in auto)
    return {'n': n, 'auto_resolution_rate': round(len(auto)/n, 3), 'accuracy_on_auto_resolved': round(auto_correct/len(auto), 3) if auto else None,
            'escalation_rate_HRR': round(len(esc)/n, 3), 'missed_escalation_rate': round(len(missed_esc)/len(gold_esc), 3) if gold_esc else None,
            'over_escalation_rate': round(len(over_esc)/n, 3), 'false_approvals_among_auto': f'{fa_auto}/{non_approvable_auto}'}
tab = pd.DataFrame({name: risk_coverage(recs) for name, recs in SYS.items()}).T
tab

,n,auto_resolution_rate,accuracy_on_auto_resolved,escalation_rate_HRR,missed_escalation_rate,over_escalation_rate,false_approvals_among_auto
2c rules (Exp 2),70,0.843,0.627,0.157,0.353,0.0,6/41
H0 RAG+M4 (Exp 9),70,0.957,0.299,0.043,0.882,0.014,0/49
H1 arithmetic,70,0.943,0.303,0.057,0.941,0.043,2/50
H3 +evidence validation,70,0.814,0.351,0.186,0.353,0.029,3/39
H5 full hybrid (Exp 12),70,0.8,0.393,0.2,0.353,0.043,2/38
workflow_v2 (Exp 18),70,0.857,0.6,0.143,0.471,0.014,7/42
R0 forced answer (Exp 12B),70,0.843,0.627,0.157,0.353,0.0,6/41
R1 escalate-unresolved (Exp 12B),70,0.343,0.833,0.657,0.059,0.429,0/23
R2 LLM residual (Exp 12B),70,0.8,0.536,0.2,0.235,0.014,0/39


## R0 vs R1 vs R2 on the shared 36-claim internally-unresolved subset: the abstention trade-off
The number that decides whether abstention is worth its human-review cost: **errors captured by escalating** vs. **correct decisions unnecessarily escalated**.

In [4]:
r0m = {r['case_id']: r for r in R0}
would_be_wrong = [cid for cid in residual_ids if r0m[cid]['predicted_decision'] != gt[cid]['expected_decision']]   # R0's forced answer, if not abstained, would be wrong
would_be_right = [cid for cid in residual_ids if cid not in would_be_wrong]
print(f'{len(residual_ids)} internally-unresolved claims: R0 (forced answer) would be WRONG on {len(would_be_wrong)}, RIGHT on {len(would_be_right)}')
print(f"\nR1 (escalates ALL {len(residual_ids)}): captures {len(would_be_wrong)} genuine would-be errors, but ALSO escalates {len(would_be_right)} claims R0 would have answered correctly -- unnecessary escalation, pure human-review cost with no error avoided")
r2m = {r['case_id']: r for r in R2}
r2_captures = sum(1 for cid in would_be_wrong if r2m[cid]['predicted_decision'] == gt[cid]['expected_decision'])
r2_unnecessary_review = sum(1 for cid in would_be_right if r2m[cid]['predicted_decision'] == 'ESCALATE')
print(f'\nR2 (LLM on the residual instead of blanket escalation): fixes {r2_captures} of the {len(would_be_wrong)} genuine would-be errors (rather than just deferring all of them to a human),')
print(f'and only escalates {r2_unnecessary_review} of the {len(would_be_right)} claims R0 would already have gotten right (vs R1s {len(would_be_right)}) -- i.e. R2 trades some human-review avoidance for solving the case itself.')
summary29 = pd.DataFrame([{'strategy':'R0 forced answer','would-be errors avoided':0,'unnecessary review of already-correct cases':0,'human_review_rate': round(risk_coverage(R0)['escalation_rate_HRR'],3)},
                          {'strategy':'R1 escalate-all-unresolved','would-be errors avoided': len(would_be_wrong),'unnecessary review of already-correct cases': len(would_be_right),'human_review_rate': round(risk_coverage(R1)['escalation_rate_HRR'],3)},
                          {'strategy':'R2 LLM-on-residual','would-be errors avoided': r2_captures,'unnecessary review of already-correct cases': r2_unnecessary_review,'human_review_rate': round(risk_coverage(R2)['escalation_rate_HRR'],3)}])
summary29

36 internally-unresolved claims: R0 (forced answer) would be WRONG on 18, RIGHT on 18

R1 (escalates ALL 36): captures 18 genuine would-be errors, but ALSO escalates 18 claims R0 would have answered correctly -- unnecessary escalation, pure human-review cost with no error avoided

R2 (LLM on the residual instead of blanket escalation): fixes 7 of the 18 genuine would-be errors (rather than just deferring all of them to a human),
and only escalates 1 of the 18 claims R0 would already have gotten right (vs R1s 18) -- i.e. R2 trades some human-review avoidance for solving the case itself.


,strategy,would-be errors avoided,unnecessary review of already-correct cases,human_review_rate
0,R0 forced answer,0,0,0.157
1,R1 escalate-all-unresolved,18,18,0.657
2,R2 LLM-on-residual,7,1,0.200


## What was done and what it means
- **The risk-coverage table makes the trade-off explicit rather than hiding it in one accuracy number.** 2c and R0 are identical, as expected (R0 *is* 2c's own decisions, reused): 84.3% auto-resolution, 62.7% accuracy on the cases it auto-resolves, and it never escalates a claim it shouldn't (0% over-escalation) but under-escalates badly (35.3% missed-escalation -- it misses more than a third of the claims that genuinely needed Finance review). The RAG/hybrid systems (H0, H1, H3, H5) auto-resolve almost everything (80-96%) but their accuracy *on those auto-resolved cases* is poor (30-39%) and missed-escalation is severe (35-94%) -- **they are not abstaining when they should, they are guessing**. `workflow_v2` sits closer to 2c (85.7% auto-resolution, 60% accuracy on auto-resolved, 47.1% missed-escalation) -- a real, if incomplete, safety improvement over the pure-LLM systems, consistent with everything found since Exp 11.
- **R1 (escalate every internally-unresolved claim) is the "safe but useless" pattern the plan explicitly warns about**: missed-escalation drops to 5.9% (its best value of any system), but auto-resolution collapses to 34.3% and Human Review Rate rises to 65.7% -- two-thirds of all claims would need a human. **R2 is the clear middle ground**: 80% auto-resolution, 20% HRR, missed-escalation nearly halved from R0's 35.3% to 23.5%, and 0/39 false approvals among its auto-resolved cases (down from R0's 6/41).
- **The headline quantification, exactly the number requested:** of the 36 internally-unresolved claims, R0's forced answer would be wrong on **18** (a 50% error rate on exactly the claims the deterministic layer itself flagged as uncertain -- strong internal validation that "unresolved" is a meaningful signal, echoing Exp 12B's earlier finding). **R1 "captures" all 18 genuine would-be errors by escalating everything, but at the cost of 18 unnecessary reviews of claims R0 would already have answered correctly** -- a 1:1 ratio of errors avoided to human time wasted. **R2 recovers 7 of the 18 genuine errors by solving them outright (rather than deferring to a human) and reduces unnecessary review to just 1 of the 18 already-correct claims** -- it trades away some of R1's blanket safety margin (11 fewer would-be-errors "caught", because it now answers instead of escalating them, and 7 of those 11 answers are actually right) in exchange for a much smaller human-review burden. **Abstention earns its cost here, but only when paired with something better than blanket escalation for the residual**: R1 alone is not a good trade (1:1 avoided-error-to-wasted-review ratio); R2 is (7 genuine errors fixed and only 1 wasted review, for a large HRR reduction versus R1).
- **False approvals among auto-resolved cases are the cleanest safety signal**, and R2 and workflow_v2/H0/R0-adjacent systems differ sharply here: R0/2c 6/41 (14.6%), workflow_v2 7/42 (16.7%, slightly worse -- consistent with Exp 18's own finding that its FAR was the highest of the deterministic-only systems), **R2 0/39 (0%)** -- the LLM-on-residual step, whatever its other limits, does not let a false approval through on this dev set.
- **Cost:** $0 -- every number here comes from predictions and the run log already on disk from Exp 2, 9, 12, 12B and 18; no new LLM calls. Total spend unchanged at $2.109 of $3.50.
- **Decision:** abstention is doing real, measurable work, but only in R2's form (abstain-then-adjudicate), not R1's (abstain-then-defer-to-human) -- the plan's own caution that "a system that escalates 100% of claims is safe but useless" is borne out numerically here (R1's 1:1 trade ratio). **The final architecture (Exp 30) should use R2's shape**: deterministic decision where conclusive, LLM adjudication (not blanket escalation) for the internally-unresolved residual, keeping human review for the residual's own genuine escalation calls rather than all of it. This is consistent with, and adds hard numbers to, Exp 12B's original finding.